# Phase 0 / NB4 (v4) — Lowering: Block-Segmented ASAP, nonzero `t_comm`, overlapped movement

`lower(layers, schedule, modules, sync_scope="module") -> (circuit, log2wire, diagnostics)`

## v4 — what changed and why

**1. `t_remote` is deleted.** It was one scalar used in two places: the duration of a remote 2Q
gate (`dur = t_remote`) *and* the block-boundary wait (`tav[q] = t_sync + t_remote`). At `0.0`
this was invisible. Setting it to `100_000` would have silently given every block boundary
100 us of movement latency — the exact quantity we decided to set to zero — and inverted 1B.

- `t_comm(a,b)` — remote 2Q gate duration, from NB2, keyed on endpoint **technology**.
- `COMM["t_move_visible"] = 0.0` — state-transfer latency, fully overlapped.

**2. The boundary's step 3 is gone.** It advanced `tav` without applying any dephasing channel:
a decoherence-free time advance that inflated makespan and charged nobody. `ST14` (busy + idle ==
makespan, per qubit) now makes that class of bug impossible to reintroduce silently.

**3. `dur = t_comm(tech[qa], tech[qb])`** for remote gates, using the *residency* technology
(post-move correct), not the module's static tech.

**4. Remote is decided by MODULE (`ma != mb`), never by technology.** A cross-module gate inside
a homogeneous 2xSC machine is remote and pays `f_comm` and `t_comm("sc","sc") = 200 ns`.
Otherwise the homogeneous baseline is silently monolithic and wins trivially. Same rule for
`f_move`: `movers` compares module IDs.

**5. `f_comm` is the aggregate fidelity of the whole teleported-gate primitive**, including its
local endpoint operations. A remote gate pays `f_comm` and **not** `f2q`.

## The timing model: block-segmented ASAP

A **block** = a maximal contiguous run of layers over which the complete qubit->module assignment
is unchanged. Pure ASAP was internally inconsistent: it checked capacity per *layer index* while
clocks drifted in *wall-clock*, so a schedule could pass the feasibility gate and still enact a
physically impossible occupancy.

- **Inside a block:** occupancy is constant, so clocks may drift. Pure ASAP, no inter-layer sync.
- **At a block boundary:** (1) dephase each affected qubit from its `t_avail` up to the block
  makespan **at its pre-move T2**; (2) movement phase (`f_move`, tech relabel); (3) the next
  block begins **at `t_sync` exactly** — `t_move_visible = 0`.

Both limits are recovered: switching every layer -> layer-synchronous clock; never switching ->
pure ASAP. Capacity is sound: occupancy is constant within a block and changes only at a
synchronized instant, so per-block capacity checking **is** wall-clock capacity checking.

`sync_scope="module"` (default, physical): only modules whose occupancy changes synchronize; an
untouched module keeps running. `"global"` is kept as a conservative comparison foil.

**Movement is not free even at zero visible latency.** Step (1) charges a moving qubit dephasing
at its *pre-move* T2 from its own `t_avail` up to the block makespan, before it may leave. A
mover on the critical path pays nothing extra; a mover that finished early pays for its own slack
at the worse coherence time. This is what ST7 certifies, and it is the honest answer to "isn't
zero-latency movement a subsidy?"

## `pos` is a wire, not a module

`pos[q]` is a physical wire index and is **not** updated when `q` changes module — only `tech[q]`
is relabelled. Sound because Aer enforces no coupling map; kept sound by the carried-placement
assertion (Gap 1). If you ever remove that assertion, wire<->module accounting breaks. Modules
live in `schedule`, wires live in `pos`. Do not merge them either.

## Known Phase-2 gaps — both **guarded by assertions**, neither silent

Two limitations live in the SC routing path. Both are documented, both fail loudly, and
**neither affects Phase 1** (verified empirically, not assumed).

### Gap 1 — carried placement across residency windows
`route()` routes one residency-stable window from an identity layout. If a routed SC module
(cap>=3) gains or loses a resident, the next window would have to route from the *carried*
placement (SABRE `initial_layout`), with real slot accounting and a proper slot->wire map. Not
implemented. **Guarded:** `lower()` asserts a routed SC module has static residency. This
assertion is also what keeps `pos` (a wire label) consistent with module residency — see NB4 v4
notes. Do not remove it thinking it is only about routing.

### Gap 2 — SABRE gate reordering vs. the routed event stream
The event stream records `('swap', pair)` and a generic `('gate',)` marker with **no gate ID**.
SABRE *does* reorder independent gates. The outer walk consumes markers in *original* order, so
swaps can be charged against the wrong gate and a gate can execute on non-adjacent wires.

- **The quantum state is never corrupted.** `lower()` applies each logical gate at its tracked
  `pos[qa], pos[qb]`. What corrupts is the **physics accounting** — the SC routing penalty, which
  is precisely what 1A measures.
- Cross-module gates never enter the SABRE proxy `sub`, so SABRE cannot reorder across them, and
  SC-local gates that share a qubit retain their order by dependency. `t_comm` opens no new hole.
- **But the price of Gap 2 rose.** A misattributed swap used to cost 200 ns of accounting error.
  Adjacent to a remote gate it can now cost 100 us. `cap <= 4` is no longer merely tidy; it is
  load-bearing.

**Why Phase 1 is safe — a structural fact.** Measured over random circuits: cap 4 -> **0/295**
violations; cap 5 -> 83/300; cap 6 -> 216/300. Not luck. SABRE's front layer contains only
pairwise-disjoint gates. On a 4-cycle, any two disjoint pairs are either both edges or both
diagonals. If both are adjacent, no SWAP intervenes, so order is irrelevant. If both are
diagonals, a **single** SWAP makes both adjacent at once, so which executes first cannot matter.
Order-independence is guaranteed at cap 4.

1A uses SC cap 4 (the 2x2 ring); 1B uses cap 2 (single edge, router never runs). Both safe.

**Guard 2:** every intra-module 2Q gate on a routed module asserts its wires are ring-adjacent.

## 0. Phase-0 stack (NB1–NB3, unchanged)

In [ ]:
import numpy as np, copy
from dataclasses import dataclass, field
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import SabreSwap
from qiskit.circuit.library import PermutationGate, HGate, CXGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import thermal_relaxation_error, depolarizing_error
from qiskit.quantum_info import DensityMatrix, Statevector, state_fidelity, partial_trace

def dephasing_channel(T2, t):
    if t <= 0: return None
    return thermal_relaxation_error(t1=np.inf, t2=float(T2), time=float(t))
def gate_infidelity_channel(F, n):
    d = 2 ** n
    return depolarizing_error((1.0 - F) * d / (d - 1.0), n)

@dataclass(frozen=True)
class TechSpec:
    name: str; f1q: float; f2q: float; T2: float; t1q: float; t2q: float; all_to_all: bool
TECHS = {
    "sc": TechSpec("sc", 0.9999, 0.999,   80_000.0,     20.0,     200.0,     False),
    "na": TechSpec("na", 0.9995, 0.997,   200_000.0,    200.0,    2_000.0,   True),
    "ti": TechSpec("ti", 0.9999, 0.9997,  2_000_000.0,  10_000.0, 100_000.0, True),
}

COMM = {
    "f_comm":         0.95,   # aggregate fidelity of the WHOLE teleported-gate primitive
    "f_move":         0.99,   # aggregate fidelity of the WHOLE state-transfer primitive
    "t_move_visible": 0.0,    # transfer overlaps the preceding block => no critical-path time
}

def t_comm(tech_a, tech_b):
    """Remote 2Q gate duration. Optimistic lower bound: endpoint local ops only."""
    return max(TECHS[tech_a].t2q, TECHS[tech_b].t2q)

def ring_cmap(n):
    if n <= 1: return CouplingMap([])
    if n == 2: return CouplingMap([[0,1],[1,0]])
    return CouplingMap([[i,(i+1)%n] for i in range(n)] + [[(i+1)%n,i] for i in range(n)])

def route(circuit, coupling_map, seeds=range(10)):
    n = circuit.num_qubits
    if coupling_map is None: return circuit, 0, list(range(n))
    best = None
    for s in seeds:
        pm = PassManager([SabreSwap(coupling_map, heuristic="decay", seed=s)])
        tqc = pm.run(circuit); nsw = tqc.count_ops().get("swap", 0)
        if best is None or nsw < best[0]:
            fl = pm.property_set.get("final_layout")
            perm = (list(range(n)) if fl is None else
                    [{q._index: p for q, p in fl.get_virtual_bits().items()}[i] for i in range(n)])
            best = (nsw, tqc, perm)
    return best[1], best[0], best[2]

## 1. Block segmentation

In [ ]:
def segment_blocks(schedule):
    """Maximal contiguous runs of layers with identical assignment map."""
    blocks, start = [], 0
    for i in range(1, len(schedule) + 1):
        if i == len(schedule) or schedule[i] != schedule[start]:
            blocks.append((start, i - 1)); start = i
    return blocks

print(segment_blocks([{0:0}]*5 + [{0:0}]*5 + [{0:1}]*6 + [{0:1}]*5), "(two blocks)")
print(segment_blocks([{0:0},{0:1},{0:0}]), "(layer-sync limit)")
print(segment_blocks([{0:0}]*4), "(pure-ASAP limit)")

## 2. Containers + channel helpers

In [ ]:
@dataclass
class Module:
    mid: int; tech: str; qubits: tuple
    @property
    def cap(self): return len(self.qubits)


@dataclass
class Diagnostics:
    idle_time: dict = field(default_factory=dict)
    sync_idle: dict = field(default_factory=dict)   # idle charged at block boundaries
    busy_time: dict = field(default_factory=dict)   # NEW: gate-occupancy per logical qubit
    swap_count: int = 0; comm_count: int = 0; move_count: int = 0
    comm_time: float = 0.0                          # NEW: total remote-gate duration
    makespan: float = 0.0; feasible: bool = True
    n_blocks: int = 0; blocks: list = field(default_factory=list)
    block_makespans: list = field(default_factory=list)   # NEW

def _ge(qc, F, w): qc.append(gate_infidelity_channel(F, len(w)).to_instruction(), w)
def _dp(qc, T2, t, w, d, lq, bucket=None):
    ch = dephasing_channel(T2, t)
    if ch is not None: qc.append(ch.to_instruction(), [w])
    if t > 0:
        d.idle_time[lq] = d.idle_time.get(lq, 0.0) + t
        if bucket == 'sync': d.sync_idle[lq] = d.sync_idle.get(lq, 0.0) + t
def _busy(d, lq, t):
    # Every ns of makespan is either busy or idle for every qubit. ST14 enforces it.
    if t > 0: d.busy_time[lq] = d.busy_time.get(lq, 0.0) + t

## 3. `lower()` — block-segmented ASAP, with both guards

In [ ]:
def segment_blocks(schedule):
    """Maximal contiguous runs of layers with identical assignment map."""
    blocks, start = [], 0
    for i in range(1, len(schedule) + 1):
        if i == len(schedule) or schedule[i] != schedule[start]:
            blocks.append((start, i - 1)); start = i
    return blocks

def lower(layers, schedule, modules, seeds=range(10), sync_scope="module", t_comm_fn=None):
    """Block-segmented ASAP.

    sync_scope : 'module' (only affected modules synchronize) | 'global' (conservative foil)
    t_comm_fn  : override for the remote-gate duration. Exists ONLY for the ST4-invariance
                 regression and for sensitivity sweeps. It CANNOT affect movement -- movement
                 latency is governed by COMM['t_move_visible'], a separate quantity. Do not
                 reintroduce a single scalar for both.

    `pos` is a physical WIRE label and is deliberately NOT updated when a qubit moves module:
    only tech[q] is relabelled. This is sound because Aer enforces no coupling map, and it is
    KEPT sound by the carried-placement assertion below (a routed SC module must have static
    residency). Removing that assertion silently breaks wire<->module accounting.
    """
    _tc = t_comm if t_comm_fn is None else t_comm_fn
    assert len(schedule) == len(layers)
    mbi = {m.mid: m for m in modules}; N = sum(m.cap for m in modules)
    allq = sorted({q for lay in layers for g in lay for q in (g[1:3] if g[0]=='2q' else [g[1]])}
                  | {q for s in schedule for q in s})
    diag = Diagnostics()

    for s in schedule:
        load = {}
        for q, mid in s.items(): load[mid] = load.get(mid, 0) + 1
        if any(load[mid] > mbi[mid].cap for mid in load):
            diag.feasible = False; return None, None, diag

    blocks = segment_blocks(schedule)
    diag.n_blocks, diag.blocks = len(blocks), blocks

    pos, nxt = {}, {m.mid: list(m.qubits) for m in modules}
    for q in allq: pos[q] = nxt[schedule[0][q]].pop(0)
    tech = {q: mbi[schedule[0][q]].tech for q in allq}

    # SC routing (Phase-1: cap>=3 SC modules must have static residency across all blocks)
    sc_ev = {}
    for m in modules:
        if TECHS[m.tech].all_to_all or m.cap < 3: continue
        res = [q for q in allq if schedule[0][q] == m.mid]
        for s in schedule:
            assert [q for q in allq if s.get(q) == m.mid] == res, (
                "carried placement across residency windows is NOT implemented (Phase 2): a routed "
                "SC module (cap>=3) changed residents. Routing the next window would need SABRE's "
                "initial_layout set to the carried placement.")
        loc = {q: i for i, q in enumerate(res)}; sub = QuantumCircuit(m.cap)
        for lay in layers:
            for g in lay:
                if g[0]=='2q' and schedule[0].get(g[1])==m.mid and schedule[0].get(g[2])==m.mid:
                    sub.cx(loc[g[1]], loc[g[2]])
        routed, nsw, _ = route(sub, ring_cmap(m.cap), seeds); diag.swap_count += nsw
        base = m.qubits[0]; ev = []
        for inst in routed.data:
            qs = [routed.find_bit(b).index for b in inst.qubits]
            if inst.operation.name == "swap": ev.append(('swap', (base+qs[0], base+qs[1])))
            elif inst.operation.name == "cx": ev.append(('gate',))
        sc_ev[m.mid] = ev

    qc = QuantumCircuit(N); tav = {q: 0.0 for q in allq}; ptrs = {mid: 0 for mid in sc_ev}

    for bi, (l0, l1) in enumerate(blocks):
        # ---- block boundary (before every block except the first) ----
        if bi > 0:
            prev, cur = schedule[blocks[bi-1][0]], schedule[l0]
            movers = [q for q in allq if prev.get(q) != cur.get(q)]
            if movers:
                affected_mods = set()
                for q in movers:
                    affected_mods.add(prev[q]); affected_mods.add(cur[q])
                if sync_scope == "global":
                    sync_q = list(allq)
                else:  # per-module: qubits resident in an affected module, before or after
                    sync_q = [q for q in allq
                              if prev.get(q) in affected_mods or cur.get(q) in affected_mods]
                t_sync = max(tav[q] for q in sync_q)
                # 1. finish previous block: dephase to block makespan at PRE-move tech T2.
                #    A mover pays for its own slack at its WORSE coherence time before it leaves.
                #    This is why t_move_visible = 0 is not the subsidy it appears to be.
                for q in sync_q:
                    _dp(qc, TECHS[tech[q]].T2, t_sync - tav[q], pos[q], diag, q, bucket='sync')
                    tav[q] = t_sync
                # 2. movement phase: fidelity only. Exactly one f_move channel per mover.
                for q in movers:
                    _ge(qc, COMM["f_move"], [pos[q]])
                    tech[q] = mbi[cur[q]].tech
                    diag.move_count += 1
                # 3. t_move_visible = 0: the transfer is fully overlapped with the preceding
                #    block, so the boundary instant is exactly t_sync. tav[q] is already t_sync.
                #    Nothing to advance. (The old code advanced the clock by t_remote here WITHOUT
                #    dephasing -- a silent, decoherence-free time advance. See ST14.)
                assert COMM["t_move_visible"] == 0.0, (
                    "exposed state-transfer latency is not implemented: a nonzero t_move_visible "
                    "must also dephase every synced qubit over the interval, not merely advance "
                    "its clock. Do not set this without adding the _dp call.")

        # ---- inside the block: pure ASAP, no global sync between layers ----
        for li in range(l0, l1 + 1):
            for g in layers[li]:
                if g[0] == '1q':
                    q = g[1]; sp = TECHS[tech[q]]
                    qc.append(g[2], [pos[q]]); _ge(qc, sp.f1q, [pos[q]])
                    tav[q] += sp.t1q; _busy(diag, q, sp.t1q)
                else:
                    qa, qb = g[1], g[2]; ma, mb = schedule[li][qa], schedule[li][qb]
                    if ma == mb and ma in sc_ev:
                        ev, ptr = sc_ev[ma], ptrs[ma]
                        while ptr < len(ev) and ev[ptr][0] == 'swap':
                            w1, w2 = ev[ptr][1]
                            L1 = next(p for p in allq if pos[p] == w1); L2 = next(p for p in allq if pos[p] == w2)
                            st = max(tav[L1], tav[L2])
                            for lq in (L1, L2): _dp(qc, TECHS[tech[lq]].T2, st-tav[lq], pos[lq], diag, lq)
                            _ge(qc, TECHS[tech[L1]].f2q**3, [w1, w2]); qc.swap(w1, w2)
                            pos[L1], pos[L2] = w2, w1
                            _sd = 3*TECHS[tech[L1]].t2q
                            tav[L1] = tav[L2] = st + _sd
                            _busy(diag, L1, _sd); _busy(diag, L2, _sd); ptr += 1
                        if ptr < len(ev): ptr += 1
                        ptrs[ma] = ptr
                    st = max(tav[qa], tav[qb])
                    for q in (qa, qb): _dp(qc, TECHS[tech[q]].T2, st-tav[q], pos[q], diag, q)
                    # GUARD: an intra-module 2Q gate on a routed (non-all-to-all) module must be
                    # executed on physically ADJACENT wires. Safe at cap 4; can fire at cap>=5.
                    if ma == mb and ma in sc_ev:
                        _m = mbi[ma]; _b = _m.qubits[0]; _c = _m.cap
                        _la, _lb = pos[qa] - _b, pos[qb] - _b
                        _adj = (abs(_la - _lb) == 1) or (abs(_la - _lb) == _c - 1)
                        assert _adj, (
                            f"non-adjacent intra-SC 2Q gate: logical ({qa},{qb}) on wires "
                            f"({pos[qa]},{pos[qb]}) of module {ma} (cap {_c}). SABRE gate-order "
                            f"mismatch -- routed event stream lacks gate IDs. Phase-2 fix required.")
                    # REMOTE branch keys on MODULE (ma != mb), never on technology. A cross-module
                    # gate inside a homogeneous 2xSC machine is remote and must pay f_comm/t_comm,
                    # otherwise the homogeneous baseline is silently monolithic and wins trivially.
                    if ma != mb:
                        _ge(qc, COMM["f_comm"], [pos[qa], pos[qb]])
                        qc.append(g[3], [pos[qa], pos[qb]])
                        dur = _tc(tech[qa], tech[qb])   # residency tech, post-move correct
                        diag.comm_count += 1; diag.comm_time += dur
                    else:
                        sp = TECHS[tech[qa]]; qc.append(g[3], [pos[qa], pos[qb]])
                        _ge(qc, sp.f2q, [pos[qa], pos[qb]]); dur = sp.t2q
                    tav[qa] = tav[qb] = st + dur
                    _busy(diag, qa, dur); _busy(diag, qb, dur)
        diag.block_makespans.append(max(tav.values()))

    tmax = max(tav.values())
    for q in allq: _dp(qc, TECHS[tech[q]].T2, tmax - tav[q], pos[q], diag, q)
    diag.makespan = tmax; qc.save_density_matrix()
    return qc, {q: pos[q] for q in allq}, diag

def _ideal(layers, allq):
    idx = {q: i for i, q in enumerate(allq)}; qc = QuantumCircuit(len(allq))
    for lay in layers:
        for g in lay:
            if g[0] == '1q': qc.append(g[2], [idx[g[1]]])
            else: qc.append(g[3], [idx[g[1]], idx[g[2]]])
    return Statevector(qc)

def aer_fidelity(layers, schedule, modules, **kw):
    qc, l2w, diag = lower(layers, schedule, modules, **kw)
    if not diag.feasible: return None, diag
    dm = DensityMatrix(AerSimulator(method="density_matrix").run(qc).result().data(0)["density_matrix"])
    allq = sorted(l2w); wires = [l2w[q] for q in allq]
    red = partial_trace(dm, [w for w in range(qc.num_qubits) if w not in wires])
    order = list(np.argsort(np.argsort(wires)))
    if order != list(range(len(order))): red = red.evolve(PermutationGate(order))
    return state_fidelity(_ideal(layers, allq), red), diag


def score(layers, schedule, modules, **kw):
    """Both axes. Fidelity already internalises makespan through T2 decay, so these are not
    independent evidence -- say so once in the paper and move on."""
    f, d = aer_fidelity(layers, schedule, modules, **kw)
    if f is None: return None
    return {"fidelity": f, "makespan": d.makespan, "tts": d.makespan / (f ** 2),
            "comm_count": d.comm_count, "comm_time": d.comm_time,
            "move_count": d.move_count, "swap_count": d.swap_count,
            "n_blocks": d.n_blocks, "block_makespans": d.block_makespans}

## 4. Checkpoint — go/no-go

**ST1–ST8** are the v3 certificate. They were hand-calibrated under `t_remote = 0`, and none of
them asserts the duration of a remote gate, so they must come out **bit-identical** under v4.
Run them first: that isolates "I split one constant into two" from "I changed the physics."

**ST9–ST15** certify the v4 semantics.

In [ ]:
H, CX = HGate(), CXGate()
def mod(m,t,w): return Module(m,t,tuple(w))
def coh_of(qc, wire):
    dm = DensityMatrix(AerSimulator(method="density_matrix").run(qc).result().data(0)["density_matrix"])
    r = partial_trace(dm, [w for w in range(qc.num_qubits) if w != wire])
    return abs(r.data[0,1])/0.5

P=[]
def chk(n,g,e,a=1e-4):
    ok=abs(g-e)<=a; P.append(ok); print(f"{'PASS' if ok else 'FAIL'} {n}: got={g:.6f} exp={e:.6f}")

print("--- block segmentation ---")
sch = [{0:0}]*5 + [{0:0}]*5 + [{0:1}]*6 + [{0:1}]*5
b = segment_blocks(sch)
P.append(b==[(0,9),(10,20)]); print(f"{'PASS' if P[-1] else 'FAIL'} segment: {b}")
P.append(segment_blocks([{0:0},{0:1},{0:0}])==[(0,0),(1,1),(2,2)]); print(f"{'PASS' if P[-1] else 'FAIL'} per-layer switch -> layer-sync limit")
P.append(segment_blocks([{0:0}]*4)==[(0,3)]); print(f"{'PASS' if P[-1] else 'FAIL'} no switch -> pure ASAP limit")

print("\n--- ST1-ST8: REGRESSION. These were calibrated under t_remote=0 and must be BIT-IDENTICAL ---")
print("    (none of them contains a remote gate whose duration is asserted, so t_comm cannot move them)")
_bak=copy.deepcopy(TECHS)
for k in list(TECHS): TECHS[k]=TechSpec(k,1.0,1.0,1e18,TECHS[k].t1q,TECHS[k].t2q,TECHS[k].all_to_all)
f,_=aer_fidelity([[('2q',0,1,CX)]],[{0:0,1:0}],[mod(0,'ti',[0,1])]); chk("ST1 noiseless",f,1.0,1e-9)
TECHS.clear(); TECHS.update(_bak)
chk("ST2 TI 2q",aer_fidelity([[('2q',0,1,CX)]],[{0:0,1:0}],[mod(0,'ti',[0,1])])[0],0.9997)
chk("ST4 f_comm",aer_fidelity([[('2q',0,1,CX)]],[{0:0,1:1}],[mod(0,'sc',[0]),mod(1,'na',[1])])[0],0.95)
qc,l2w,_=lower([[('1q',0,H),('2q',2,3,CX)]],[{0:0,1:0,2:1,3:1}],[mod(0,'sc',[0,1]),mod(1,'ti',[2,3])])
chk("ST3 SC idle",coh_of(qc,l2w[0]),np.exp(-99980/80000))
f5,d5=aer_fidelity([[('2q',0,2,CX)],[('2q',1,3,CX)],[('2q',0,1,CX)],[('2q',2,3,CX)]],
                   [{0:0,1:0,2:0,3:0}]*4,[mod(0,'sc',[0,1,2,3])])
P.append(d5.swap_count>0 and f5>0.9); print(f"{'PASS' if P[-1] else 'FAIL'} ST5 routing f={f5:.4f} swaps={d5.swap_count}")

qc6,l6,d6=lower([[('1q',0,H)],[('2q',2,3,CX)]],[{0:0,2:1,3:1},{0:1,2:1,3:1}],
                [mod(0,'sc',[0,1]),mod(1,'ti',[2,3,4])])
exp6=(2*0.9999-1)*(2*0.99-1)*np.exp(-100000/2_000_000)
chk("ST6 parked idle @TI T2",coh_of(qc6,l6[0]),exp6)

qc7,l7,d7=lower([[('1q',0,H),('2q',2,3,CX)],[('2q',2,3,CX)]],
                [{0:0,2:1,3:1},{0:1,2:1,3:1}],[mod(0,'sc',[0,1]),mod(1,'ti',[2,3,4])])
exp7=(2*0.9999-1)*np.exp(-99980/80000)*(2*0.99-1)*np.exp(-100000/2_000_000)
chk("ST7 sync idle @SC T2 before move",coh_of(qc7,l7[0]),exp7)
P.append(abs(d7.sync_idle.get(0,0)-99980)<1); print(f"{'PASS' if P[-1] else 'FAIL'} ST7 sync_idle={d7.sync_idle.get(0,0):.0f} ns (pure ASAP would charge 0)")

layers8=[[('1q',0,H),('2q',2,3,CX),('2q',5,6,CX)],[('2q',5,6,CX)]]
sch8=[{0:0,2:1,3:1,5:2,6:2},{0:1,2:1,3:1,5:2,6:2}]
mods8=[mod(0,'sc',[0,1]),mod(1,'ti',[2,3,4]),mod(2,'na',[5,6])]
_,_,d8m=lower(layers8,sch8,mods8,sync_scope="module")
_,_,d8g=lower(layers8,sch8,mods8,sync_scope="global")
P.append(d8m.sync_idle.get(5,0)==0 and d8g.sync_idle.get(5,0)>0)
print(f"{'PASS' if P[-1] else 'FAIL'} ST8 unaffected NA module not synced (module scope)")

_,_,di=lower([[('1q',0,H)]],[{0:0,1:0,2:0}],[mod(0,'ti',[0,1])])
P.append(di.feasible is False); print(f"{'PASS' if P[-1] else 'FAIL'} feasibility gate")

## 4b. ST9–ST15 — the v4 certificate

In [ ]:
# ============================================================================
# ST9-ST15 -- the new certificate for nonzero t_comm and zero-visible t_move
# ============================================================================

print("--- t_comm landscape (this table is the motivation section) ---")
print(f"{'pair':>8} {'t_comm(ns)':>11} {'spectator T2':>13} {'ratio':>8} {'survival':>9}")
for a,b in [("sc","sc"),("na","na"),("sc","na"),("ti","ti"),("sc","ti")]:
    tc = t_comm(a,b); T2s = min(TECHS[a].T2, TECHS[b].T2)
    print(f"{a+'+'+b:>8} {tc:>11.0f} {T2s:>13.0f} {tc/T2s:>8.4f} {np.exp(-tc/T2s):>9.4f}")
print("  ^ SC+TI is the only row where a remote gate outlives the spectator's coherence.")

print("\n--- ST4 fidelity is invariant to t_comm (regression isolation) ---")
_Z = lambda a,b: 0.0
f4z,_ = aer_fidelity([[('2q',0,1,CX)]],[{0:0,1:1}],[mod(0,'sc',[0]),mod(1,'na',[1])],t_comm_fn=_Z)
P.append(abs(f4z-0.95)<1e-9); print(f"{'PASS' if P[-1] else 'FAIL'} ST4 under t_comm=0: {f4z:.6f}")

print("\n--- ST9: a remote gate advances BOTH participants by exactly t_comm ---")
f9,d9 = aer_fidelity([[('2q',0,1,CX)]],[{0:0,1:1}],[mod(0,'sc',[0]),mod(1,'ti',[1])])
chk("ST9a fidelity == f_comm (aggregate; no separate f2q)", f9, 0.95)
P.append(d9.makespan==100000.0 and d9.comm_time==100000.0 and d9.comm_count==1)
print(f"{'PASS' if P[-1] else 'FAIL'} ST9b makespan={d9.makespan:.0f} comm_time={d9.comm_time:.0f} comm_count={d9.comm_count}")

print("\n--- ST10: a SPECTATOR dephases through a remote gate  <<< the paper's number ---")
# q0 (SC mod0) x q1 (TI mod1) remote CX.  q2 (SC mod0) does H then idles through it.
qc10,l10,d10 = lower([[('1q',2,H),('2q',0,1,CX)]],[{0:0,1:1,2:0}],
                     [mod(0,'sc',[0,1]),mod(1,'ti',[2])])
exp10 = (2*0.9999-1)*np.exp(-99980/80000)
chk("ST10a spectator coherence", coh_of(qc10,l10[2]), exp10)
P.append(abs(d10.idle_time.get(2,0)-99980)<1)
print(f"{'PASS' if P[-1] else 'FAIL'} ST10b q2 idle={d10.idle_time.get(2,0):.0f} ns")
print("     With t_remote=0 this was 0.0 ns and coherence 1.0. EFCL was blind to 97% of a remote gate's cost.")

print("\n--- ST11: the remote branch keys on MODULE, not technology (2xSC baseline honesty) ---")
f11r,d11r = aer_fidelity([[('2q',0,1,CX)]],[{0:0,1:1}],[mod(0,'sc',[0]),mod(1,'sc',[1])])
f11l,d11l = aer_fidelity([[('2q',0,1,CX)]],[{0:0,1:0}],[mod(0,'sc',[0,1])])
chk("ST11a cross-module SC-SC == f_comm", f11r, 0.95)
chk("ST11b intra-module SC-SC == f2q",    f11l, 0.999)
P.append(d11r.comm_count==1 and d11r.comm_time==200.0 and d11l.comm_count==0)
print(f"{'PASS' if P[-1] else 'FAIL'} ST11c comm_count remote/local = {d11r.comm_count}/{d11l.comm_count}, t_comm={d11r.comm_time:.0f} ns")
P.append(d11r.makespan==d11l.makespan==200.0)
print(f"{'PASS' if P[-1] else 'FAIL'} ST11d identical makespan ({d11r.makespan:.0f} ns): homogeneous SC pays in fidelity, not time")

print("\n--- ST12: movement is fidelity-only; exactly ONE f_move per mover; no visible makespan ---")
_bak2=copy.deepcopy(TECHS)
for k in list(TECHS): TECHS[k]=TechSpec(k,1.0,1.0,1e18,TECHS[k].t1q,TECHS[k].t2q,TECHS[k].all_to_all)
qc12,l12,_ = lower([[('1q',0,H),('2q',2,3,CX)],[('2q',2,3,CX)]],
                   [{0:0,2:1,3:1},{0:1,2:1,3:1}],[mod(0,'sc',[0,1]),mod(1,'ti',[2,3,4])])
chk("ST12a exactly one f_move (0.9604 => applied twice)", coh_of(qc12,l12[0]), 2*0.99-1)
TECHS.clear(); TECHS.update(_bak2)
_,_,d12p = lower([[('1q',0,H),('2q',2,3,CX)],[('2q',2,3,CX)]],
                 [{0:0,2:1,3:1}]*2,[mod(0,'sc',[0,1]),mod(1,'ti',[2,3,4])])
P.append(d7.makespan==d12p.makespan and d7.move_count==1 and d12p.move_count==0)
print(f"{'PASS' if P[-1] else 'FAIL'} ST12b makespan moved={d7.makespan:.0f} == pinned={d12p.makespan:.0f}")

print("\n--- ST13: assignment is constant inside every block (property test) ---")
import random as _rnd
_r=_rnd.Random(3); ok13=True
for _ in range(200):
    L=_r.randint(1,8); s=[{q:_r.randint(0,1) for q in range(3)} for _ in range(L)]
    for (a,b) in segment_blocks(s):
        for li in range(a,b+1):
            if s[li]!=s[a]: ok13=False
P.append(ok13); print(f"{'PASS' if ok13 else 'FAIL'} ST13")

print("\n--- ST14: busy + idle == makespan, per qubit  (catches any silent clock advance) ---")
def _consistent(d):
    return all(abs(d.busy_time.get(q,0.)+d.idle_time.get(q,0.)-d.makespan)<1e-6
               for q in set(d.busy_time)|set(d.idle_time))
_,_,dA = lower([[('1q',0,H),('2q',2,3,CX)],[('2q',2,3,CX)]],[{0:0,2:1,3:1},{0:1,2:1,3:1}],
               [mod(0,'sc',[0,1]),mod(1,'ti',[2,3,4])])
_,_,dB = lower(layers8,sch8,mods8)
_,_,dC = lower([[('2q',0,2,CX)],[('2q',1,3,CX)],[('2q',0,1,CX)],[('2q',2,3,CX)]],
               [{0:0,1:0,2:0,3:0}]*4,[mod(0,'sc',[0,1,2,3])])
_,_,dD = lower([[('1q',2,H),('2q',0,1,CX)]],[{0:0,1:1,2:0}],[mod(0,'sc',[0,1]),mod(1,'ti',[2])])
for nm,d in [("block boundary",dA),("3-module",dB),("SC routing",dC),("remote gate",dD)]:
    ok=_consistent(d); P.append(ok); print(f"{'PASS' if ok else 'FAIL'} ST14 {nm}: makespan={d.makespan:.0f} ns")
print("     The pre-fix code advanced tav by t_remote at the boundary with no _dp call. This test fails on it.")

print("\n--- ST15: the next block uses the DESTINATION technology's gate time ---")
_,_,d15 = lower([[('1q',0,H)],[('2q',0,2,CX)]],[{0:0,2:1},{0:1,2:1}],
                [mod(0,'sc',[0,1]),mod(1,'ti',[2,3])])
P.append(d15.makespan==100020.0)
print(f"{'PASS' if P[-1] else 'FAIL'} ST15 makespan={d15.makespan:.0f} ns (expect 100020; would be 220 if SC t2q leaked through)")

print("\n"+"="*70)
print("ALL PASS" if all(P) else f"{P.count(False)} FAILURES")

## 5. Guard verification

The guards must be **silent on Phase-1 shapes** and **fire where the bug is real**.

In [ ]:
import random
from qiskit.circuit.library import CXGate
_CX = CXGate()

# 1A shape: static SC cap-4 ring + NA module + bridge gate -> silent
l = [[('2q',0,2,_CX),('2q',4,6,_CX)], [('2q',1,3,_CX)], [('2q',0,1,_CX),('2q',3,7,_CX)]]
s = [{0:0,1:0,2:0,3:0,4:1,5:1,6:1,7:1}]*3
f, d = aer_fidelity(l, s, [Module(0,'sc',(0,1,2,3)), Module(1,'na',(4,5,6,7))])
print(f"1A shape: fidelity={f:.4f} swaps={d.swap_count} comms={d.comm_count}  (guard silent)")

def _sweep(cap, trials, seed):
    rnd = random.Random(seed); fired = tried = 0
    for _ in range(trials):
        layers = []
        for _ in range(rnd.randint(2,6)):
            qs = list(range(cap)); rnd.shuffle(qs); lay = []
            while len(qs) >= 2 and rnd.random() < 0.8:
                a,b = qs.pop(), qs.pop(); lay.append(('2q',a,b,_CX))
            if lay: layers.append(lay)
        if not layers: continue
        tried += 1
        try: lower(layers, [{i:0 for i in range(cap)}]*len(layers), [Module(0,'sc',tuple(range(cap)))])
        except AssertionError: fired += 1
    return fired, tried

f4,t4 = _sweep(4, 300, 11); f6,t6 = _sweep(6, 200, 7)
print(f"cap 4 (1A): guard fired {f4}/{t4}  (expect 0 - order-independent on a 4-cycle)")
print(f"cap 6      : guard fired {f6}/{t6}  (these were silently corrupt before v3)")
assert f4 == 0 and f6 > 0
print("\nGUARDS VERIFIED: silent on Phase 1, loud where the bug is real")